# 02 — Bronze Layer (Auto Loader → VARIANT)

Uses Auto Loader (`cloudFiles`) in batch mode to read raw JSON files from the
Volume landing zone and write VARIANT-based bronze Delta tables.

Each bronze table has:
- `data VARIANT` — the full raw JSON object (semi-structured)
- `_ingestion_timestamp` — when the row was loaded
- `_source_file` — which file the row came from

Silver layer will extract typed columns using `data:field::type` syntax.

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "ticketmaster_demo")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
VOLUME_NAME   = "raw_data"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"

print(f"Bronze schema: {UC_CATALOG}.{BRONZE_SCHEMA}")
print(f"Volume path:   {VOLUME_PATH}")

Bronze schema: alexander_booth.ticketmaster_demo_bronze
Volume path:   /Volumes/alexander_booth/ticketmaster_demo_bronze/raw_data


In [2]:
ENTITIES = ["events", "venues", "attractions", "classifications"]

for entity in ENTITIES:
    source_path     = f"{VOLUME_PATH}/{entity}/"
    checkpoint_path = f"{VOLUME_PATH}/_checkpoints/{entity}"
    target_table    = f"{UC_CATALOG}.{BRONZE_SCHEMA}.{entity}_raw"

    print(f"\nLoading {entity} from {source_path}")
    print(f"  Checkpoint: {checkpoint_path}")
    print(f"  Target:     {target_table}")

    # Auto Loader reads JSON files, parses each array element as a row,
    # and stores the full object as a VARIANT column.
    # inferColumnTypes=true preserves nested structure so VARIANT paths work.
    query = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", checkpoint_path + "/_schema")
        .option("multiLine", "true")
        .load(source_path)
        .selectExpr(
            "PARSE_JSON(TO_JSON(STRUCT(*))) AS data",
            "current_timestamp() AS _ingestion_timestamp",
            "_metadata.file_path AS _source_file"
        )
        .writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(target_table)
    )

    query.awaitTermination()
    print(f"  Done: {entity}_raw")


Loading events from /Volumes/alexander_booth/ticketmaster_demo_bronze/raw_data/events/
  Checkpoint: /Volumes/alexander_booth/ticketmaster_demo_bronze/raw_data/_checkpoints/events
  Target:     alexander_booth.ticketmaster_demo_bronze.events_raw
  Done: events_raw

Loading venues from /Volumes/alexander_booth/ticketmaster_demo_bronze/raw_data/venues/
  Checkpoint: /Volumes/alexander_booth/ticketmaster_demo_bronze/raw_data/_checkpoints/venues
  Target:     alexander_booth.ticketmaster_demo_bronze.venues_raw
  Done: venues_raw

Loading attractions from /Volumes/alexander_booth/ticketmaster_demo_bronze/raw_data/attractions/
  Checkpoint: /Volumes/alexander_booth/ticketmaster_demo_bronze/raw_data/_checkpoints/attractions
  Target:     alexander_booth.ticketmaster_demo_bronze.attractions_raw
  Done: attractions_raw

Loading classifications from /Volumes/alexander_booth/ticketmaster_demo_bronze/raw_data/classifications/
  Checkpoint: /Volumes/alexander_booth/ticketmaster_demo_bronze/raw_dat

In [3]:
# Show sample data and row counts
print("\n" + "=" * 60)
print("BRONZE LAYER SUMMARY")
print("=" * 60)

for entity in ENTITIES:
    table = f"{UC_CATALOG}.{BRONZE_SCHEMA}.{entity}_raw"
    count = spark.table(table).count()
    print(f"  {table}: {count:,} rows")

print("\nSample events_raw:")
spark.sql(f"""
    SELECT data:name::string AS name, data:id::string AS id,
           data:dates.start.localDate::string AS event_date,
           _ingestion_timestamp
    FROM {UC_CATALOG}.{BRONZE_SCHEMA}.events_raw
    LIMIT 5
""").show(truncate=False)


BRONZE LAYER SUMMARY
  alexander_booth.ticketmaster_demo_bronze.events_raw: 4,800 rows
  alexander_booth.ticketmaster_demo_bronze.venues_raw: 1,600 rows
  alexander_booth.ticketmaster_demo_bronze.attractions_raw: 1,600 rows
  alexander_booth.ticketmaster_demo_bronze.classifications_raw: 34 rows

Sample events_raw:
+--------------------------------------------------------------------+---------------------+----------+-----------------------+
|name                                                                |id                   |event_date|_ingestion_timestamp   |
+--------------------------------------------------------------------+---------------------+----------+-----------------------+
|IBM DataPower Online | Corporate Training                           |LvZ18p4sVcRDRC8ZnrK0y|2017-07-13|2026-03-19 20:55:19.902|
|Welcome to the Universe!                                            |LvZ189dbEEKvP5Yv7f1_4|2019-01-10|2026-03-19 20:55:19.902|
|EPISODE 4: Meditation & Stretch w/ DeAndre